In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px

# Parametros

In [ ]:
ambiente = 'dev'
costa = 'Matamoros'
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 2
RANDOM_SEED = 0
SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
MODEL_FEATURES = ['wind_speed_ms', 'wave_height_m', 'wave_period_s', 'wave_steepness']
SORT_FEATURES = ["wave_height_m", "wind_speed_ms", "wave_steepness"]

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_clasification/wave_clasification_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    model_path = f'{base_path}/wave_clasification/wave_clasification_{{}}.pkl'
    

# Obtener datos

In [3]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{costa}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [4]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}

data_sample = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

In [5]:
fig = px.scatter_matrix(
    data_sample, 
    dimensions=MODEL_FEATURES
)
fig.update_layout(
    title=f'Correlación de variables de la costa {costa}',
    width=1200, 
    height=700
)
fig.show()

In [6]:
correlation_matrix = data_sample[MODEL_FEATURES].corr()
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto")
fig.update_layout(
    title=f'Matriz de correlación de variables de la costa {costa}'
)
fig.show()

# Escalar

In [7]:
scaler_path = scaler_path.format(costa)
scaler = joblib.load(scaler_path)

In [8]:
scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

In [9]:
scaled_df

,wind_speed_ms,wind_cos_direction,wind_sin_direction,wave_height_m,wave_cos_direction,wave_sin_direction,wave_period_s,wave_energy,wave_steepness
0,0.845115,-1.636183,-0.417159,0.006459,1.553187,0.124252,-0.287046,-0.152898,0.475400
1,1.708934,-1.669895,-0.356902,0.463760,1.692354,-0.179523,-0.256005,0.198757,1.244130
2,2.160299,-1.676463,-0.344624,1.009003,1.765954,-0.376414,-0.090457,0.705376,1.888871
3,2.253685,-1.690911,-0.316668,1.536658,1.798738,-0.475025,0.168213,1.279515,2.219507
4,2.171972,-1.747536,-0.195586,2.539203,1.830538,-0.578600,0.975264,2.622137,2.070721
...,...,...,...,...,...,...,...,...,...
6173,0.206979,-1.686825,1.767213,0.199932,1.999703,-3.163332,-0.421554,-0.007531,1.070546
6174,0.436552,-1.784751,1.545451,0.164755,2.022652,-2.977692,-0.380167,-0.036721,0.921759
6175,-0.182129,-1.915806,0.450243,-0.046307,2.041011,-1.758622,-0.235312,-0.187962,0.310082
6176,-0.380574,-1.736298,-0.221276,-0.134249,1.951183,-1.080258,-0.162884,-0.250274,0.053838


# Gaussian Mixture

In [10]:
from sklearn.mixture import GaussianMixture

## Parametros

In [11]:
N_CLUSTERS = 6

In [12]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [13]:
data_sample["cluster"] = gmm.fit_predict(scaled_df[MODEL_FEATURES])
data_sample["cluster_probability"] = gmm.predict_proba(scaled_df[MODEL_FEATURES]).max(axis=1)

In [14]:
data_sample

,coast_name,datetime,wind_speed_ms,wind_cos_direction,wind_sin_direction,wave_height_m,wave_cos_direction,wave_sin_direction,wave_period_s,wave_energy,wave_steepness,cluster,cluster_probability
0,Matamoros,2018-01-01 00:00:00,8.84,-0.7980,-0.6027,1.25,0.6904,0.7234,5.08,1968.729980,0.0484,1,0.826553
1,Matamoros,2018-01-01 01:00:00,11.06,-0.8211,-0.5708,1.51,0.7753,0.6316,5.11,2870.830078,0.0577,1,0.999735
2,Matamoros,2018-01-01 02:00:00,12.22,-0.8256,-0.5643,1.82,0.8202,0.5721,5.27,4170.459961,0.0655,1,0.722359
3,Matamoros,2018-01-01 03:00:00,12.46,-0.8355,-0.5495,2.12,0.8402,0.5423,5.52,5643.299805,0.0695,3,0.999469
4,Matamoros,2018-01-01 06:00:00,12.25,-0.8743,-0.4854,2.69,0.8596,0.5110,6.30,9087.530273,0.0677,3,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6173,Matamoros,2018-12-31 17:00:00,7.20,-0.8327,0.5537,1.36,0.9628,-0.2701,4.95,2341.639893,0.0556,1,0.999854
6174,Matamoros,2018-12-31 18:00:00,7.79,-0.8998,0.4363,1.34,0.9768,-0.2140,4.99,2266.760010,0.0538,1,0.999922
6175,Matamoros,2018-12-31 21:00:00,6.20,-0.9896,-0.1435,1.22,0.9880,0.1544,5.13,1878.780029,0.0464,1,0.619517
6176,Matamoros,2018-12-31 22:00:00,5.69,-0.8666,-0.4990,1.17,0.9332,0.3594,5.20,1718.930054,0.0433,5,0.916214


In [ ]:
cluster_summary = (
    data_sample.groupby("cluster")[MODEL_FEATURES]
    .mean()
    .sort_values(SORT_FEATURES)
)

cluster_order = {
    old_cluster: new_cluster + 1
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

data_sample["sea_state_level"] = data_sample["cluster"].map(cluster_order)

EXTREME_FEATURES = ['wind_speed_ms', 'wave_height_m']
data_sample['mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(0.99)
for feature in EXTREME_FEATURES[1:]:
    data_sample['mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(0.99)

data_sample.loc[data_sample['mask_extremo'], 'sea_state_level'] = 7

In [16]:
data_sample['sea_state_level'].value_counts(normalize=True)*100

sea_state_level
1    25.530179
2    19.412724
5    19.119086
4    15.995106
3    10.872757
6     8.409462
7     0.660685
Name: proportion, dtype: float64

In [17]:
data_sample

,coast_name,datetime,wind_speed_ms,wind_cos_direction,wind_sin_direction,wave_height_m,wave_cos_direction,wave_sin_direction,wave_period_s,wave_energy,wave_steepness,cluster,cluster_probability,sea_state_level,mask_extremo
0,Matamoros,2017-01-01 01:00:00,9.62,0.0859,0.9963,1.81,-0.8827,0.4699,5.93,4132.520020,0.0516,1,0.856599,5,False
1,Matamoros,2017-01-01 02:00:00,9.42,0.1255,0.9921,1.80,-0.8991,0.4377,5.88,4068.879883,0.0520,1,0.909504,5,False
2,Matamoros,2017-01-01 03:00:00,9.03,0.1334,0.9911,1.76,-0.9065,0.4221,5.86,3914.790039,0.0513,1,0.862226,5,False
3,Matamoros,2017-01-01 05:00:00,8.76,0.0687,0.9976,1.71,-0.9181,0.3964,5.80,3681.270020,0.0508,1,0.845666,5,False
4,Matamoros,2017-01-01 06:00:00,8.66,0.1464,0.9892,1.69,-0.9233,0.3842,5.77,3586.639893,0.0507,1,0.856598,5,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12255,Matamoros,2018-12-31 17:00:00,7.20,-0.8327,0.5537,1.36,0.9628,-0.2701,4.95,2341.639893,0.0556,1,0.999923,5,False
12256,Matamoros,2018-12-31 18:00:00,7.79,-0.8998,0.4363,1.34,0.9768,-0.2140,4.99,2266.760010,0.0538,1,0.999958,5,False
12257,Matamoros,2018-12-31 21:00:00,6.20,-0.9896,-0.1435,1.22,0.9880,0.1544,5.13,1878.780029,0.0464,1,0.783508,5,False
12258,Matamoros,2018-12-31 22:00:00,5.69,-0.8666,-0.4990,1.17,0.9332,0.3594,5.20,1718.930054,0.0433,5,0.859103,4,False


In [18]:
sea_state_names = {
    1: "Mar calmado",
    2: "Mar suave",
    3: "Mar dinámico",
    4: "Mar agitado",
    5: "Mar fuerte",
    6: "Mar peligroso",
    7: "Mar extremo"
}

data_sample["sea_state"] = data_sample["sea_state_level"].map(sea_state_names)

In [19]:
data_sample['sea_state'].value_counts(normalize=True)*100

sea_state
Mar calmado      25.530179
Mar suave        19.412724
Mar fuerte       19.119086
Mar agitado      15.995106
Mar dinámico     10.872757
Mar peligroso     8.409462
Mar extremo       0.660685
Name: proportion, dtype: float64

In [20]:
fig = px.scatter_3d(
    data_sample, 
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_height_m',
    color='sea_state'
)
fig.update_layout(
    title=f'Muestra datos de la costa {costa}',
    uirevision='constant',
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Altura de la ola (m)',
        aspectmode='cube'
    ),
    width=800, 
    height=700
)
fig.update_traces(marker=dict(size=3))
fig.show()

# Random Forest

In [21]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)

In [22]:
TARGET = "sea_state_level"

In [23]:
X = data_sample[MODEL_FEATURES]
y = data_sample[TARGET].astype(int)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [ ]:
y_pred = rf_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)

cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.9637846655791191
Balanced accuracy: 0.9693767306967958
              precision    recall  f1-score   support

           1      0.968     0.966     0.967       783
           2      0.971     0.971     0.971       595
           3      0.947     0.967     0.957       333
           4      0.960     0.941     0.951       490
           5      0.978     0.964     0.971       586
           6      0.930     0.977     0.953       258
           7      1.000     1.000     1.000        20

    accuracy                          0.964      3065
   macro avg      0.965     0.969     0.967      3065
weighted avg      0.964     0.964     0.964      3065

[[756  11   8   7   1   0   0]
 [ 15 578   0   0   0   2   0]
 [  2   0 322   6   0   3   0]
 [  8   1   8 461   9   3   0]
 [  0   5   0   5 565  11   0]
 [  0   0   2   1   3 252   0]
 [  0   0   0   0   0   0  20]]


In [ ]:
feature_importance = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

,feature,importance
1,wave_height_m,0.380580
3,wave_steepness,0.269936
2,wave_period_s,0.212926
0,wind_speed_ms,0.136557


In [ ]:
pd.DataFrame(rf_classifier.predict(X), columns=['predicted_sea_state_level']).value_counts(normalize=True)*100

predicted_sea_state_level
1                            25.212072
2                            19.543230
5                            18.980424
4                            15.823817
3                            10.986949
6                             8.792822
7                             0.660685
Name: proportion, dtype: float64

In [ ]:
model_path = model_path.format(costa)
joblib.dump(rf_classifier, model_path)